In [1]:
import sys
import os
import pandas as pd
from pandapower.timeseries.data_sources.frame_data import DFData
from pandaprosumer.create import create_empty_prosumer_container
from pandaprosumer.create import create_period
from pandaprosumer.create_controlled import create_controlled_const_profile
from pandaprosumer.create_controlled import create_controlled_ice_chp
from pandaprosumer.create_controlled import create_controlled_heat_storage
from pandaprosumer.create_controlled import create_controlled_heat_demand
from pandaprosumer.mapping import GenericMapping
from pandaprosumer.run_time_series import run_timeseries
import matplotlib.pyplot as plt

In [2]:
size_kw = 1400
altitude_m = 0
fuel = 'ng'
name = 'example_chp'

In [3]:
start = '2020-01-01 00:00:00'
end = '2020-01-02 00:00:00'
time_resolution_s = 900         # 15 min
frequency = '15min'


# HEAT DEMAND
demand_data = pd.read_excel('data/input_chp_2consumers.xlsx')
dur = pd.date_range(start, end, freq=frequency, tz='utc')
demand_data.index = dur
demand_input = DFData(demand_data)

In [4]:
dur = pd.date_range(start=start, end=end, freq=frequency, tz='utc')
demand_data.index = dur
demand_input = DFData(demand_data)

print(demand_data)

                                          time  cycle  t_intake_k  \
2020-01-01 00:00:00+00:00             00:00:00      2         273   
2020-01-01 00:15:00+00:00             00:15:00      2         273   
2020-01-01 00:30:00+00:00             00:30:00      2         273   
2020-01-01 00:45:00+00:00             00:45:00      2         273   
2020-01-01 01:00:00+00:00             01:00:00      2         273   
2020-01-01 01:15:00+00:00             01:15:00      2         273   
2020-01-01 01:30:00+00:00             01:30:00      2         273   
2020-01-01 01:45:00+00:00             01:45:00      2         273   
2020-01-01 02:00:00+00:00             02:00:00      2         273   
2020-01-01 02:15:00+00:00             02:15:00      2         273   
2020-01-01 02:30:00+00:00             02:30:00      2         273   
2020-01-01 02:45:00+00:00             02:45:00      2         273   
2020-01-01 03:00:00+00:00             03:00:00      2         273   
2020-01-01 03:15:00+00:00         

In [5]:
prosumer = create_empty_prosumer_container()

period = create_period(prosumer, time_resolution_s, start, end, 'utc', 'default')

In [6]:
input_columns = ['q_demand1_kw', 'q_demand2_kw', 'cycle', 't_intake_k']
output_columns = ['q_demand1_cp_kw', 'q_demand2_cp_kw', 'cycle_cp', 't_intake_cp_k']
cp_index = create_controlled_const_profile(
    prosumer, input_columns, output_columns, demand_input, period)

# ice chp controller
ice_chp_index = create_controlled_ice_chp(prosumer, size_kw, fuel, altitude_m, name, level=1, order=0)

# heat deman 1 controller
heat_demand1_index = create_controlled_heat_demand(prosumer, scaling=1.0, level=1, order=1)

# heat deman 2 controller
heat_demand2_index = create_controlled_heat_demand(prosumer, scaling=1.0, level=1, order=2)

TypeError: Unexpected type for 'period_index' (expected <class 'int'> but found <class 'pandapower.timeseries.data_sources.frame_data.DFData'>)